# Getting started: Modeling *Gaia* astrometry data

**harv** is a JAX-native Python package for inferring Keplerian orbital parameters of
binary star or star–exoplanet systems. `harv` provides a rejection sampler that
implements analytic marginalization over linear parameters to improve the efficiency of
the sampling, along with extensions to support continued sampling with MCMC. `harv` is
an evolution of the rejection sampler implemented in [*The
Joker*](https://github.com/adrn/thejoker), so see {doc}`../../concepts` and the [original
paper](https://arxiv.org/abs/2209.14359) for more details about the method.

This tutorial covers basic functionality around modeling *Gaia* astrometry data, including:
- Creating a {py:class}`~harv.GaiaAstrometryData` object,
- Setting up a {py:class}`~harv.HarvPrior` with the default prior distributions and parameterization,
- Running the {py:class}`~harv.RejectionSampler` to get posterior samples, and
- Visualizing the results.

Other tutorials demonstrate more advanced functionality:
- {doc}`Customizing the prior distributions <1-customize-prior>`
- {doc}`Continuing the posterior sampling with numpyro <3-mcmc-continue>`
- {doc}`Extending the model with extra noise terms, trends, or instrumental offsets <2-model-extensions>`
- {doc}`Modeling SB2 systems <4-sb2>`

Let's start with the imports we'll need:

In [ ]:
import astropy.table as at
import jax
import numpyro.distributions as dist
import quaxed.numpy as jnp
from unxt import Q

import harv
import harv.models as hm

jax.config.update("jax_enable_x64", True)

%matplotlib inline

## Loading *Gaia* astrometry data

The main data container for *Gaia* astrometry data in `harv` is {py:class}`harv.GaiaAstrometryData`, which stores barycentric observation times, measured proper motions, and uncertainties. These are all specified as `unxt.Quantity` objects with explicit physical units, which is like the more familiar `astropy.units.Quantity` but with support for JAX transformations. See the [unxt documentation](https://unxt.readthedocs.io/en/latest/) for more details. In `unxt`, it is standard to use the short-hand alias `Q` for `unxt.Quantity`, which we do for brevity below. Since `jax.numpy` doesn't know how to handle `unxt.Quantity` objects, we use `quaxed.numpy` (imported as `jnp`) for any mathematical operations - see the [unxt documentation](https://unxt.readthedocs.io/en/latest/#jax-functions) or the {doc}`sharp bits page <../../sharp-bits>` for more information about working with `unxt` and `jax`.

For this tutorial, we will use simulated *Gaia* astrometry ...

https://dace.unige.ch/openData/?record=10.82180/dace-gaia-ohp

In [ ]:
# tbl = at.Table.read("../data/gaia-DACE-ohp/target_6.csv")
tbl = at.Table.read("../data/gaia-DACE-ohp/target_7.csv")

fdtype = jnp.float64
data = harv.GaiaAstrometryData(
    time=Q(tbl["obs_time_tcb"].astype(fdtype), "day"),
    al_position=Q(tbl["centroid_pos_al"].astype(fdtype), "mas"),
    al_position_err=Q(tbl["centroid_pos_error_al"].astype(fdtype), "mas"),
    scan_angle=Q(tbl["scan_pos_angle"].astype(fdtype), "rad"),
    parallax_factor=tbl["parallax_factor_al"].astype(fdtype),
)
data

Let's start by plotting the data:

In [ ]:
_ = data.plot(relative_to_t_ref=True)

TODO: words about the data

## Specifying the prior distributions

With data in hand, our next task is to set up the prior probability distribution for rejection sampling. The default prior for radial velocity data can be initialized using the `default_prior()` method on {py:class}`~harv.models.StandardRV`, which sets up a log-uniform period prior, the Kipping (2013) eccentricity prior, along with uniform priors on the phase and argument of periastron. The linear parameters (RV semi-amplitude and systemic velocity) are marginalized out analytically, with a Gaussian prior whose width (in semi-amplitude) scales with period and eccentricity to keep it approximately constant in companion mass.

Here, `period_min` and `period_max` set the minimum and maximum periods for the default log-uniform period prior, `sigma_K0` sets the scale factor for the Gaussian prior on the RV semi-amplitude at the reference period `P0` (by default 1 year, but this can be configured by changing `P0`), and `sigma_v0` sets the (fixed) width of the Gaussian prior on the systemic velocity. The default values for these parameters are set to be broadly appropriate for stellar companions, but they can be customized as needed (for example, to be more appropriate for exoplanet companions) - see the next tutorial for more details.

In [ ]:
prior = hm.StandardGaiaAstrometry().default_prior(
    period_min=Q(1e2, "day"),
    period_max=Q(1e4, "day"),
    sigma_pos=Q(100, "mas"),
    sigma_vtan=Q(100, "km/s"),
    sigma_a0=Q(1, "AU"),
    P0=Q(300, "day"),
    parallax=harv.QD(dist.Normal(34.3, 0.05), "mas"),
)

Let's see the breakdown of which parameters are included in the nonlinear and linear priors using the default prior setup:

In [ ]:
print(prior.nonlinear_priors.keys())
print(prior.linear_priors.keys())

The prior has four nonlinear parameters (period, eccentricity, orbital phase, argument of periastron).  The linear parameters are `rv_semiamp` or $K$ and `v_sys` or $v_0$.

## Running the rejection sampler

With data and prior now specified, we can create a {py:class}`harv.RejectionSampler` to manage running the rejection sampling:

In [ ]:
sampler = harv.RejectionSampler(
    prior,
    harv.GaiaAstrometryModel(),
    marginalized_names=("ra0", "dec0", "pmra", "pmdec", "semi_major_axis"),
)

And then we can call `.run()` to run the sampler. The default specification for the sampler requires the number of prior samples to draw (`n_prior_samples`) and the maximum number of posterior samples to return (`max_posterior_samples`). You can also set a random number seed (`seed`) for reproducibility.

In [ ]:
samples = sampler.run(
    data,
    n_prior_samples=10_000_000,
    max_posterior_samples=1024,
    seed=123,
    return_logprobs=True,
)
samples

In [ ]:
axes = samples.wrap_angles().plot_corner(
    ["period", "eccentricity", "semi_major_axis", "parallax", "pmra", "pmdec"],
    visuals={"scatter": {"alpha": 0.75, "s": 8}},
)

In [ ]:
axes = samples.wrap_angles().plot_corner(
    ["arg_peri", "cos_i", "lon_asc_node", "phase_peri", "ra0", "dec0"],
    visuals={"scatter": {"alpha": 0.75, "s": 8}},
)

## Working with and visualizing the returned posterior samples

I find it useful to first look at the returned posterior samples to get a sense of the structure of the posterior and the types of orbits that are consistent with the data. As a first visualization, we can use the ``Samples.plot_corner()`` method to make a corner plot of the posterior samples:



In [ ]:
# axes = samples.plot_corner(
#     ["period", "eccentricity", "rv_semiamp", "v_sys"],
#     visuals={"scatter": {"alpha": 0.75, "s": 2}},
# )
# for ax in axes.viz["plot"].isel(col_index=0).values.flat:
#     if ax is not None:
#         ax.set_xscale("log")

Right away we can notice a few things:
- First, the period distribution is highly multi-modal, with many well-separated modes corresponding to different orbital periods that are consistent with the data. This is a common feature of RV posteriors when the data are sparse or low signal-to-noise, and this is exactly why the rejection sampling method is so powerful for RV data: it can efficiently map out these complex posteriors in a way that MCMC methods would struggle with.
- Second, the RV semi-amplitude `rv_semiamp` has positive and negative values. This is a byproduct of the Gaussian prior we adopt on the semiamplitude in order to do the analytic marginalization. The negative values of `rv_semiamp` are equivalent to positive values with a 180 degree shift in the argument of periastron `arg_peri`. We can use the `Samples.wrap_angles()` method to wrap the angles to a more standard range (e.g. 0 to 360 degrees) and to keep physical parameters positive (e.g., semi-amplitude or, for astrometry, semimajor axis) if desired:

In [ ]:
# wrapped_samples = samples.wrap_angles()
# axes = wrapped_samples.plot_corner(
#     ["period", "eccentricity", "rv_semiamp", "v_sys"],
#     visuals={"scatter": {"alpha": 0.75, "s": 2}},
# )
# for ax in axes.viz["plot"].isel(col_index=0).values.flat:
#     if ax is not None:
#         ax.set_xscale("log")

Another useful visualization is to inspect the best-fit orbit for a single posterior sample using {py:func}`harv.plot.plot_gaia_astrometry`. This function plots a single sample, so we pass the maximum a posteriori sample via {py:meth}`~harv.samplers.Samples.map_sample` (you could instead pick a sample by index, e.g. `samples[i]`). The left panel shows the sky-projected photocenter orbit, and the right panel shows the along-scan position residuals — the observed positions minus the full model (orbit + parallax + proper motion + zero-point):

In [ ]:
fig = harv.plot.plot_gaia_astrometry(
    samples.map_sample(),
    data=data,
)

## Next steps

With only a few observations, the posterior is highly multi-modal in orbital period. 

In other settings, we may want to infer orbital parameters for data with more observations, higher signal-to-noise, or with a more constrained period. In these cases, the posterior may be less multi-modal and more amenable to MCMC sampling. Or, we may want to extend our model to add a more sophisticated noise model. Or, we may have data from multiple instruments or surveys that have instrumental / calibration offsets. We explore these cases in the next few tutorials.